# End-to-End MLOps Pipeline for House Price Prediction
## Exploratory Data Analysis (EDA) Notebook

**Objective:** Explore the property characteristics, analyze distributions, inspect correlations with `SalePrice`, and identify preprocessing requirements for machine learning model development.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Aesthetic styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
print('Libraries imported successfully.')

### 1. Load Raw Dataset

In [2]:
data_path = Path('../data/raw/house_prices.csv')
if not data_path.exists():
    data_path = Path('data/raw/house_prices.csv')

df = pd.read_csv(data_path)
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

### 2. Dataset Summary and Missing Values

In [3]:
print("=== Missing Value Analysis ===")
missing = df.isnull().sum()
print(missing[missing > 0])

df.describe().T[['mean', 'std', 'min', '50%', 'max']]

### 3. Target Distribution: `SalePrice`

In [4]:
plt.figure(figsize=(10, 5))
sns.histplot(df['SalePrice'], kde=True, color='#1f77b4', bins=35)
plt.title('House Sale Price Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Sale Price ($ USD)')
plt.ylabel('Property Count')
plt.gca().xaxis.set_major_formatter('${x:,.0f}')
plt.show()

### 4. Correlation Analysis: Numerical Features vs `SalePrice`

In [5]:
numeric_cols = df.select_dtypes(include=[np.number]).columns
corr_matrix = df[numeric_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='Blues', square=True, cbar_kws={'label': 'Pearson Correlation'})
plt.title('Correlation Heatmap with SalePrice', fontsize=14, fontweight='bold')
plt.show()

### 5. Key Driver: Living Area & Quality vs Price

In [6]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# GrLivArea vs Price
sns.scatterplot(data=df, x='GrLivArea', y='SalePrice', hue='OverallQual', palette='viridis', ax=ax1, alpha=0.8)
ax1.set_title('Living Area vs. Sale Price by Quality', fontweight='bold')
ax1.set_xlabel('Ground Living Area (sq ft)')
ax1.set_ylabel('Sale Price ($)')
ax1.yaxis.set_major_formatter('${x:,.0f}')

# Boxplot by Overall Quality
sns.boxplot(data=df, x='OverallQual', y='SalePrice', ax=ax2, palette='Blues')
ax2.set_title('Sale Price by Overall Quality Rating', fontweight='bold')
ax2.set_xlabel('Overall Quality (1-10)')
ax2.set_ylabel('Sale Price ($)')
ax2.yaxis.set_major_formatter('${x:,.0f}')

plt.tight_layout()
plt.show()

### 6. Categorical Analysis: Neighborhood Price Variance

In [7]:
plt.figure(figsize=(12, 6))
order = df.groupby('Neighborhood')['SalePrice'].median().sort_values(ascending=False).index
sns.boxplot(data=df, x='Neighborhood', y='SalePrice', order=order, palette='crest')
plt.title('House Price Distribution Across Neighborhoods (Sorted by Median)', fontsize=14, fontweight='bold')
plt.xlabel('Neighborhood')
plt.ylabel('Sale Price ($ USD)')
plt.xticks(rotation=45)
plt.gca().yaxis.set_major_formatter('${x:,.0f}')
plt.tight_layout()
plt.show()

### 7. Key Findings & Preprocessing Strategy
- **Feature Importance:** Living area (`GrLivArea`), `OverallQual`, and `TotalBsmtSF` show strong positive linear and non-linear correlation with house prices.
- **Imputation Strategy:** Numerical columns with occasional missing values (`TotalBsmtSF`, `GarageCars`) must be imputed with median strategy to protect against skew.
- **Categorical Features:** Location (`Neighborhood`), dwelling style, and central air require one-hot encoding with `handle_unknown='ignore'` for robust inference.
- **Scaling:** Standard scaling is essential for linear regression algorithms while tree-based models (Random Forest, Gradient Boosting, XGBoost) leverage raw non-linear splits.